# Model Training — Teacher–Student Knowledge Distillation

This notebook implements the full pipeline:

| Section | Description |
|---|---|
| **D** | Teacher Model — Wav2Vec2-XLSR + AASIST + OC-Softmax |
| **E** | Teacher Soft Predictions |
| **F** | Student Model — Frozen Wav2Vec2-XLSR + AASIST-L |
| **G** | Knowledge Distillation Training |
| **H** | Final Prediction |
| **I** | Evaluation — EER, min-tDCF, Accuracy, Precision, Recall, F1, Confusion Matrix |

## 1. Install Dependencies

In [ ]:
!pip install torch torchaudio transformers soundfile numpy scipy scikit-learn matplotlib seaborn tqdm --quiet

## 2. Imports

In [ ]:
import os
import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model, Wav2Vec2Processor
from pathlib import Path
from tqdm import tqdm
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, roc_curve)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Force CPU ────────────────────────────────────────────────────────────────
device = torch.device('cpu')
print(f'Device : {device}')

## 3. Configuration

Edit the paths below to match your project structure.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
AUGMENTED_DIR    = Path('../data/Augmented/LA')
PREPROCESSED_DIR = Path('../data/Preprocessed/LA')
PROTOCOL_DIR     = Path('../data/raw/LA/ASVspoof2019_LA_cm_protocols')
FEATURE_DIR      = Path('../data/features')          # pre-extracted features saved here
MODEL_DIR        = Path('../models')

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Audio ────────────────────────────────────────────────────────────────────
SAMPLE_RATE  = 16000
MAX_SAMPLES  = 64600      # ~4 seconds at 16kHz (standard in anti-spoofing)

# ── Model ────────────────────────────────────────────────────────────────────
WAV2VEC2_MODEL   = 'facebook/wav2vec2-large-xlsr-53'
WAV2VEC2_DIM     = 1024   # output feature dimension of Wav2Vec2-XLSR

# ── Training ─────────────────────────────────────────────────────────────────
BATCH_SIZE       = 8
TEACHER_EPOCHS   = 10
STUDENT_EPOCHS   = 10
LEARNING_RATE    = 1e-4
KD_TEMPERATURE   = 3.0    # softens teacher predictions
KD_ALPHA         = 0.5    # weight: 0.5 * CE + 0.5 * KD loss

print('Configuration set.')

## 4. Protocol Parsing

Parse the ASVspoof2019 protocol files to get filenames and labels.

Protocol format: `speaker_id  filename  -  attack_type  label`

In [ ]:
def parse_protocol(protocol_path):
    """
    Returns a dict: {filename: label}
    label → 1 = bonafide, 0 = spoof
    """
    data = {}
    with open(protocol_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            filename = parts[1]                        # e.g. LA_T_1000137
            label    = 1 if parts[4] == 'bonafide' else 0
            data[filename] = label
    return data


train_labels = parse_protocol(PROTOCOL_DIR / 'ASVspoof2019.LA.cm.train.trn.txt')
dev_labels   = parse_protocol(PROTOCOL_DIR / 'ASVspoof2019.LA.cm.dev.trl.txt')
eval_labels  = parse_protocol(PROTOCOL_DIR / 'ASVspoof2019.LA.cm.eval.trl.txt')

print(f'Train : {len(train_labels)} files')
print(f'Dev   : {len(dev_labels)}   files')
print(f'Eval  : {len(eval_labels)}  files')
print(f'  Bonafide (train): {sum(v for v in train_labels.values())}')
print(f'  Spoof    (train): {sum(1-v for v in train_labels.values())}')

## 5. Feature Extraction — Wav2Vec2-XLSR

Wav2Vec2-XLSR is used as the **front-end** for both teacher and student.
Features are pre-extracted and saved to disk to avoid re-running the large model every epoch.

> This step is slow on CPU. Run once and skip on subsequent runs.

In [ ]:
def load_audio(path, max_samples=MAX_SAMPLES):
    """Load and pad/truncate audio to fixed length."""
    x, sr = sf.read(str(path), dtype='float32')
    if x.ndim > 1:
        x = x.mean(axis=1)
    if len(x) > max_samples:
        x = x[:max_samples]
    else:
        x = np.pad(x, (0, max_samples - len(x)))
    return x


def extract_and_save_features(audio_dir, label_dict, split_name, wav2vec2, processor):
    """
    Extract Wav2Vec2 features for all files in a split and save as .npy.
    Skips files already extracted.
    """
    save_dir = FEATURE_DIR / split_name
    save_dir.mkdir(parents=True, exist_ok=True)

    wav2vec2.eval()
    filenames = list(label_dict.keys())

    for fname in tqdm(filenames, desc=f'Extracting {split_name}'):
        out_path = save_dir / f'{fname}.npy'
        if out_path.exists():
            continue

        # Find the audio file
        audio_path = audio_dir / 'flac' / f'{fname}.flac'
        if not audio_path.exists():
            continue

        audio = load_audio(audio_path)

        inputs = processor(
            audio,
            sampling_rate=SAMPLE_RATE,
            return_tensors='pt',
            padding=True
        )

        with torch.no_grad():
            features = wav2vec2(**inputs).last_hidden_state  # [1, T, 1024]

        np.save(str(out_path), features.squeeze(0).numpy())  # [T, 1024]

    print(f'{split_name} features saved to {save_dir}')

In [ ]:
# Load Wav2Vec2-XLSR (shared by teacher and student)
print('Loading Wav2Vec2-XLSR...')
processor = Wav2Vec2Processor.from_pretrained(WAV2VEC2_MODEL)
wav2vec2  = Wav2Vec2Model.from_pretrained(WAV2VEC2_MODEL)
wav2vec2.eval()
print('Wav2Vec2-XLSR loaded.')

In [ ]:
# Extract features for all splits
# Train uses Augmented data; Dev and Eval use Preprocessed (clean)
extract_and_save_features(
    AUGMENTED_DIR    / 'ASVspoof2019_LA_train', train_labels, 'train', wav2vec2, processor)

extract_and_save_features(
    PREPROCESSED_DIR / 'ASVspoof2019_LA_dev',  dev_labels,   'dev',   wav2vec2, processor)

extract_and_save_features(
    PREPROCESSED_DIR / 'ASVspoof2019_LA_eval', eval_labels,  'eval',  wav2vec2, processor)

## 6. Dataset

In [ ]:
class ASVspoofDataset(Dataset):
    """
    Loads pre-extracted Wav2Vec2 features and labels.
    Optionally loads teacher soft predictions for KD training.
    """
    def __init__(self, split, label_dict, soft_preds=None):
        self.feature_dir = FEATURE_DIR / split
        self.soft_preds  = soft_preds     # dict: {filename: soft_pred_array}

        # Keep only files that have extracted features
        self.samples = [
            (fname, label)
            for fname, label in label_dict.items()
            if (self.feature_dir / f'{fname}.npy').exists()
        ]
        print(f'{split}: {len(self.samples)} samples loaded')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fname, label = self.samples[idx]
        feat = np.load(str(self.feature_dir / f'{fname}.npy'))  # [T, 1024]

        # Pad or truncate to fixed T=200 frames
        T = 200
        if feat.shape[0] >= T:
            feat = feat[:T]
        else:
            pad = np.zeros((T - feat.shape[0], feat.shape[1]), dtype=np.float32)
            feat = np.concatenate([feat, pad], axis=0)

        item = {
            'feature' : torch.tensor(feat, dtype=torch.float32),
            'label'   : torch.tensor(label, dtype=torch.long),
            'filename': fname
        }

        if self.soft_preds and fname in self.soft_preds:
            item['soft_pred'] = torch.tensor(self.soft_preds[fname], dtype=torch.float32)

        return item


train_dataset = ASVspoofDataset('train', train_labels)
dev_dataset   = ASVspoofDataset('dev',   dev_labels)
eval_dataset  = ASVspoofDataset('eval',  eval_labels)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader    = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False)
eval_loader   = DataLoader(eval_dataset,  batch_size=BATCH_SIZE, shuffle=False)

## D. Teacher Model — AASIST Backend

AASIST classifies audio as bona fide or spoof by learning **spectral and temporal spoofing patterns** using graph attention over the Wav2Vec2 features.

In [ ]:
class AASISTBackend(nn.Module):
    """
    AASIST backend.
    Input : [batch, T, 1024]  Wav2Vec2-XLSR features
    Output: [batch, 2]        logits (bona fide / spoof)

    Architecture
    ------------
    Linear projection → 2× Spectro-Temporal Attention blocks
    → Graph-style pooling → Classifier
    """
    def __init__(self, input_dim=1024, d_model=256, num_heads=8,
                 ff_dim=512, num_layers=2, dropout=0.1):
        super().__init__()

        self.proj = nn.Linear(input_dim, d_model)

        # Spectro-temporal attention layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Graph-style attention pooling
        self.attn_pool = nn.Linear(d_model, 1)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        # x: [batch, T, 1024]
        x = self.proj(x)                          # [batch, T, d_model]
        x = self.transformer(x)                   # [batch, T, d_model]

        # Attention pooling over time
        weights = torch.softmax(self.attn_pool(x), dim=1)  # [batch, T, 1]
        x = (x * weights).sum(dim=1)              # [batch, d_model]

        return self.classifier(x)                 # [batch, 2]


teacher_model = AASISTBackend().to(device)
total_params  = sum(p.numel() for p in teacher_model.parameters())
print(f'Teacher (AASIST) parameters: {total_params:,}')

## 7. OC-Softmax Loss

OC-Softmax pulls **bona fide** samples toward a learned centre and pushes **spoof** samples away, improving separation between real and fake speech.

In [ ]:
class OCSoftmaxLoss(nn.Module):
    """
    One-Class Softmax Loss for anti-spoofing.
    Ref: Zhang et al., One-class Learning Towards Synthetic Voice Spoofing Detection.

    - r_real : target radius for bona fide samples (pull inward)
    - r_fake : target radius for spoof samples    (push outward)
    - alpha  : scaling factor
    """
    def __init__(self, feat_dim=2, r_real=0.9, r_fake=0.5, alpha=20.0):
        super().__init__()
        self.r_real  = r_real
        self.r_fake  = r_fake
        self.alpha   = alpha
        self.center  = nn.Parameter(torch.randn(feat_dim))

    def forward(self, logits, labels):
        """
        logits : [batch, 2]
        labels : [batch]  — 1 = bonafide, 0 = spoof
        """
        # Normalise logits and centre
        logits_norm = F.normalize(logits, dim=1)
        center_norm = F.normalize(self.center.unsqueeze(0), dim=1)

        # Cosine similarity to centre
        cos_sim = (logits_norm * center_norm).sum(dim=1)  # [batch]

        # OC-Softmax margin loss
        loss_real = torch.clamp(self.r_real - cos_sim, min=0)[labels == 1].mean()
        loss_fake = torch.clamp(cos_sim - self.r_fake, min=0)[labels == 0].mean()

        # Handle edge case: all samples in batch are same class
        loss_real = loss_real if not torch.isnan(loss_real) else torch.tensor(0.0)
        loss_fake = loss_fake if not torch.isnan(loss_fake) else torch.tensor(0.0)

        return self.alpha * (loss_real + loss_fake)


oc_softmax = OCSoftmaxLoss(feat_dim=2).to(device)
print('OC-Softmax loss defined.')

## D. Train Teacher Model

In [ ]:
teacher_optimizer = torch.optim.Adam(
    list(teacher_model.parameters()) + list(oc_softmax.parameters()),
    lr=LEARNING_RATE
)

teacher_history = {'train_loss': [], 'dev_loss': [], 'dev_acc': []}

print('Training Teacher Model...')
for epoch in range(1, TEACHER_EPOCHS + 1):

    # ── Train ────────────────────────────────────────────────────────────────
    teacher_model.train()
    train_loss = 0.0

    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/{TEACHER_EPOCHS} [Teacher Train]'):
        features = batch['feature'].to(device)   # [B, T, 1024]
        labels   = batch['label'].to(device)     # [B]

        teacher_optimizer.zero_grad()
        logits = teacher_model(features)         # [B, 2]

        # OC-Softmax + standard CE combined
        loss = oc_softmax(logits, labels) + F.cross_entropy(logits, labels)
        loss.backward()
        teacher_optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ── Validate ─────────────────────────────────────────────────────────────
    teacher_model.eval()
    dev_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for batch in dev_loader:
            features = batch['feature'].to(device)
            labels   = batch['label'].to(device)
            logits   = teacher_model(features)
            dev_loss += F.cross_entropy(logits, labels).item()
            preds     = logits.argmax(dim=1)
            correct  += (preds == labels).sum().item()
            total    += labels.size(0)

    dev_loss /= len(dev_loader)
    dev_acc   = correct / total

    teacher_history['train_loss'].append(train_loss)
    teacher_history['dev_loss'].append(dev_loss)
    teacher_history['dev_acc'].append(dev_acc)

    print(f'Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | '
          f'Dev Loss: {dev_loss:.4f} | Dev Acc: {dev_acc:.4f}')

print('\nTeacher training complete.')

## E. Teacher Soft Predictions

The trained teacher generates **soft predictions** on the training set. These show confidence per class (e.g. `[0.08, 0.92]`) and are used to guide student training.

In [ ]:
teacher_model.eval()
soft_predictions = {}   # {filename: soft_pred [2,]}

# Run teacher on the train split to collect soft predictions
temp_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)

with torch.no_grad():
    for batch in tqdm(temp_loader, desc='Generating soft predictions'):
        features   = batch['feature'].to(device)
        filenames  = batch['filename']
        logits     = teacher_model(features)                      # [B, 2]
        soft_preds = F.softmax(logits / KD_TEMPERATURE, dim=1)   # [B, 2]

        for fname, pred in zip(filenames, soft_preds):
            soft_predictions[fname] = pred.cpu().numpy()

print(f'Soft predictions generated for {len(soft_predictions)} training samples.')
# Example: show one prediction
sample_key = list(soft_predictions.keys())[0]
print(f'  Example — {sample_key}: bonafide={soft_predictions[sample_key][1]:.3f}, '
      f'spoof={soft_predictions[sample_key][0]:.3f}')

## F. Student Model — AASIST-L Backend

AASIST-L is a **lightweight** version of AASIST. The Wav2Vec2-XLSR front-end is **frozen** — only AASIST-L is trained.

In [ ]:
class AASISTLBackend(nn.Module):
    """
    AASIST-L (Lightweight) backend.
    Same architecture as AASIST but with smaller dimensions:
      d_model : 256 → 128
      heads   : 8   → 4
      ff_dim  : 512 → 256
    """
    def __init__(self, input_dim=1024, d_model=128, num_heads=4,
                 ff_dim=256, num_layers=2, dropout=0.1):
        super().__init__()

        self.proj = nn.Linear(input_dim, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        x       = self.proj(x)
        x       = self.transformer(x)
        weights = torch.softmax(self.attn_pool(x), dim=1)
        x       = (x * weights).sum(dim=1)
        return self.classifier(x)


student_model = AASISTLBackend().to(device)

teacher_params = sum(p.numel() for p in teacher_model.parameters())
student_params = sum(p.numel() for p in student_model.parameters())

print(f'Teacher (AASIST)   parameters: {teacher_params:,}')
print(f'Student (AASIST-L) parameters: {student_params:,}')
print(f'Compression ratio : {teacher_params / student_params:.1f}x')
print('NOTE: Wav2Vec2-XLSR front-end is frozen (not fine-tuned during student training)')

## G. Knowledge Distillation Loss

The student learns from two signals:
- **Hard label loss** (CE): learn the correct class
- **KD loss** (KL divergence): learn from teacher's confidence scores

$$\mathcal{L} = \alpha \cdot \mathcal{L}_{CE} + (1 - \alpha) \cdot T^2 \cdot \mathcal{L}_{KL}$$

In [ ]:
def knowledge_distillation_loss(student_logits, teacher_soft_preds,
                                 hard_labels, temperature=KD_TEMPERATURE, alpha=KD_ALPHA):
    """
    Combined hard-label CE loss + KL divergence KD loss.

    Parameters
    ----------
    student_logits    : [B, 2]  raw student outputs
    teacher_soft_preds: [B, 2]  teacher softmax outputs (with temperature)
    hard_labels       : [B]     ground-truth labels
    temperature       : float   softens predictions (higher = softer)
    alpha             : float   weight for CE loss (1-alpha for KD loss)
    """
    # Hard label cross-entropy
    ce_loss = F.cross_entropy(student_logits, hard_labels)

    # KD loss: KL divergence between student and teacher soft predictions
    student_soft = F.log_softmax(student_logits / temperature, dim=1)
    kd_loss      = F.kl_div(student_soft, teacher_soft_preds,
                             reduction='batchmean') * (temperature ** 2)

    return alpha * ce_loss + (1 - alpha) * kd_loss


print('KD loss function defined.')
print(f'  alpha={KD_ALPHA} (CE weight) | temperature={KD_TEMPERATURE}')

## G. Train Student Model

In [ ]:
# Rebuild train dataset with soft predictions attached
train_dataset_kd = ASVspoofDataset('train', train_labels, soft_preds=soft_predictions)
train_loader_kd  = DataLoader(train_dataset_kd, batch_size=BATCH_SIZE, shuffle=True)

student_optimizer = torch.optim.Adam(student_model.parameters(), lr=LEARNING_RATE)

student_history = {'train_loss': [], 'dev_loss': [], 'dev_acc': []}

print('Training Student Model with Knowledge Distillation...')
for epoch in range(1, STUDENT_EPOCHS + 1):

    # ── Train ────────────────────────────────────────────────────────────────
    student_model.train()
    train_loss = 0.0

    for batch in tqdm(train_loader_kd, desc=f'Epoch {epoch}/{STUDENT_EPOCHS} [Student Train]'):
        features    = batch['feature'].to(device)
        labels      = batch['label'].to(device)
        soft_preds  = batch['soft_pred'].to(device)  # teacher predictions

        student_optimizer.zero_grad()
        logits = student_model(features)

        loss = knowledge_distillation_loss(logits, soft_preds, labels)
        loss.backward()
        student_optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader_kd)

    # ── Validate ─────────────────────────────────────────────────────────────
    student_model.eval()
    dev_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for batch in dev_loader:
            features = batch['feature'].to(device)
            labels   = batch['label'].to(device)
            logits   = student_model(features)
            dev_loss += F.cross_entropy(logits, labels).item()
            preds     = logits.argmax(dim=1)
            correct  += (preds == labels).sum().item()
            total    += labels.size(0)

    dev_loss /= len(dev_loader)
    dev_acc   = correct / total

    student_history['train_loss'].append(train_loss)
    student_history['dev_loss'].append(dev_loss)
    student_history['dev_acc'].append(dev_acc)

    print(f'Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | '
          f'Dev Loss: {dev_loss:.4f} | Dev Acc: {dev_acc:.4f}')

print('\nStudent training complete.')

## H. Final Prediction

The **student model** is the final prediction model. It takes audio, extracts features using Wav2Vec2-XLSR, and predicts bona fide or spoof.

In [ ]:
def predict(audio_path, wav2vec2, processor, student_model):
    """
    Predict whether an audio file is bona fide or spoof.

    Returns
    -------
    label : str   — 'bonafide' or 'spoof'
    score : float — confidence score for bonafide (higher = more likely real)
    """
    audio  = load_audio(audio_path)
    inputs = processor(audio, sampling_rate=SAMPLE_RATE, return_tensors='pt')

    with torch.no_grad():
        features = wav2vec2(**inputs).last_hidden_state  # [1, T, 1024]

        # Pad/truncate
        T = 200
        feat = features[0]
        if feat.shape[0] >= T:
            feat = feat[:T]
        else:
            pad  = torch.zeros(T - feat.shape[0], feat.shape[1])
            feat = torch.cat([feat, pad], dim=0)

        logits = student_model(feat.unsqueeze(0))        # [1, 2]
        probs  = F.softmax(logits, dim=1)[0]             # [2]

    pred  = 'bonafide' if probs[1] > probs[0] else 'spoof'
    score = probs[1].item()   # bonafide confidence
    return pred, score


print('predict() function ready.')
print('Usage: pred, score = predict("path/to/audio.flac", wav2vec2, processor, student_model)')

## I. Evaluation

Evaluate the student model on the **eval set** using:
- **EER** and **min-tDCF** — standard ASVspoof anti-spoofing metrics
- **Accuracy, Precision, Recall, F1** — classification metrics
- **Confusion Matrix** — visual breakdown of predictions

In [ ]:
def compute_eer(bonafide_scores, spoof_scores):
    """
    Equal Error Rate — threshold where FAR == FRR.
    Returns EER as a percentage.
    """
    scores = np.concatenate([bonafide_scores, spoof_scores])
    labels = np.concatenate([np.ones(len(bonafide_scores)),
                              np.zeros(len(spoof_scores))])
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    fnr         = 1 - tpr
    eer         = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    return eer * 100, fpr, tpr


def compute_min_tdcf(bonafide_scores, spoof_scores,
                      p_spoof=0.05, c_cm_miss=1.0, c_cm_fa=10.0):
    """
    Simplified min-tDCF for standalone CM system.
    Based on ASVspoof2019 evaluation protocol.
    """
    scores = np.concatenate([bonafide_scores, spoof_scores])
    labels = np.concatenate([np.ones(len(bonafide_scores)),
                              np.zeros(len(spoof_scores))])

    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr

    # tDCF = C_miss * P_spoof * FNR + C_fa * (1-P_spoof) * FPR
    tdcf = (c_cm_miss * p_spoof * fnr) + (c_cm_fa * (1 - p_spoof) * fpr)

    # Normalise by the minimum cost of a trivial system
    tdcf_norm = tdcf / min(c_cm_miss * p_spoof, c_cm_fa * (1 - p_spoof))

    return float(np.min(tdcf_norm))

In [ ]:
# ── Collect predictions on eval set ──────────────────────────────────────────
student_model.eval()

all_preds, all_labels, all_scores = [], [], []
bonafide_scores, spoof_scores     = [], []

with torch.no_grad():
    for batch in tqdm(eval_loader, desc='Evaluating on Eval Set'):
        features = batch['feature'].to(device)
        labels   = batch['label'].to(device)

        logits = student_model(features)
        probs  = F.softmax(logits, dim=1)            # [B, 2]
        preds  = logits.argmax(dim=1)                # [B]

        bonafide_score = probs[:, 1].cpu().numpy()   # confidence for bonafide

        for score, label in zip(bonafide_score, labels.cpu().numpy()):
            if label == 1:
                bonafide_scores.append(score)
            else:
                spoof_scores.append(score)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_scores.extend(bonafide_score)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# ── Compute metrics ───────────────────────────────────────────────────────────
eer, fpr, tpr   = compute_eer(np.array(bonafide_scores), np.array(spoof_scores))
min_tdcf        = compute_min_tdcf(np.array(bonafide_scores), np.array(spoof_scores))
accuracy        = accuracy_score(all_labels, all_preds)
precision       = precision_score(all_labels, all_preds, zero_division=0)
recall          = recall_score(all_labels, all_preds, zero_division=0)
f1              = f1_score(all_labels, all_preds, zero_division=0)
cm              = confusion_matrix(all_labels, all_preds)

print('=' * 50)
print('         EVALUATION RESULTS (Eval Set)')
print('=' * 50)
print(f'  EER          : {eer:.2f}%')
print(f'  min-tDCF     : {min_tdcf:.4f}')
print(f'  Accuracy     : {accuracy:.4f}')
print(f'  Precision    : {precision:.4f}')
print(f'  Recall       : {recall:.4f}')
print(f'  F1-Score     : {f1:.4f}')
print('=' * 50)

## 14. Save Student Model

In [ ]:
# Save the full student model
save_path = MODEL_DIR / 'student_model.keras'
torch.save({
    'model_state_dict'    : student_model.state_dict(),
    'optimizer_state_dict': student_optimizer.state_dict(),
    'student_history'     : student_history,
    'teacher_history'     : teacher_history,
    'eval_metrics': {
        'eer'       : eer,
        'min_tdcf'  : min_tdcf,
        'accuracy'  : accuracy,
        'precision' : precision,
        'recall'    : recall,
        'f1'        : f1
    }
}, str(save_path))

print(f'Student model saved → {save_path}')

# To reload:
# checkpoint = torch.load('models/student_model.keras')
# student_model.load_state_dict(checkpoint['model_state_dict'])

## 15. Training History & Evaluation Graphs

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Training History & Evaluation Results', fontsize=16, fontweight='bold')

epochs_t = range(1, TEACHER_EPOCHS + 1)
epochs_s = range(1, STUDENT_EPOCHS + 1)

# ── Plot 1: Teacher Loss ──────────────────────────────────────────────────────
axes[0, 0].plot(epochs_t, teacher_history['train_loss'], 'b-o', label='Train Loss')
axes[0, 0].plot(epochs_t, teacher_history['dev_loss'],   'r-o', label='Dev Loss')
axes[0, 0].set_title('D. Teacher Model — Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# ── Plot 2: Teacher Accuracy ──────────────────────────────────────────────────
axes[0, 1].plot(epochs_t, teacher_history['dev_acc'], 'g-o', label='Dev Accuracy')
axes[0, 1].set_title('D. Teacher Model — Dev Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_ylim(0, 1)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# ── Plot 3: Student Loss ──────────────────────────────────────────────────────
axes[0, 2].plot(epochs_s, student_history['train_loss'], 'b-o', label='Train Loss')
axes[0, 2].plot(epochs_s, student_history['dev_loss'],   'r-o', label='Dev Loss')
axes[0, 2].set_title('G. Student Model (KD) — Loss')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Loss')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# ── Plot 4: Student Accuracy ──────────────────────────────────────────────────
axes[1, 0].plot(epochs_s, student_history['dev_acc'], 'g-o', label='Dev Accuracy')
axes[1, 0].set_title('G. Student Model (KD) — Dev Accuracy')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].set_ylim(0, 1)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# ── Plot 5: ROC Curve with EER ────────────────────────────────────────────────
axes[1, 1].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (EER={eer:.2f}%)')
axes[1, 1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[1, 1].scatter([eer/100], [1 - eer/100], color='red', zorder=5, s=80, label='EER point')
axes[1, 1].set_title('I. ROC Curve — EER')
axes[1, 1].set_xlabel('False Positive Rate')
axes[1, 1].set_ylabel('True Positive Rate')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# ── Plot 6: Confusion Matrix ──────────────────────────────────────────────────
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 2],
            xticklabels=['Spoof', 'Bonafide'],
            yticklabels=['Spoof', 'Bonafide'])
axes[1, 2].set_title('I. Confusion Matrix')
axes[1, 2].set_xlabel('Predicted')
axes[1, 2].set_ylabel('Actual')

plt.tight_layout()
plt.savefig(str(MODEL_DIR / 'training_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Graphs saved → models/training_results.png')

In [ ]:
# ── Final Metrics Summary ─────────────────────────────────────────────────────
metrics = {
    'EER (%)' : f'{eer:.2f}',
    'min-tDCF': f'{min_tdcf:.4f}',
    'Accuracy': f'{accuracy:.4f}',
    'Precision': f'{precision:.4f}',
    'Recall'  : f'{recall:.4f}',
    'F1-Score': f'{f1:.4f}'
}

fig, ax = plt.subplots(figsize=(6, 3))
ax.axis('off')
table_data = [[k, v] for k, v in metrics.items()]
table = ax.table(cellText=table_data, colLabels=['Metric', 'Value'],
                 cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.5, 2)
plt.title('I. Final Evaluation Metrics (Eval Set)', fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(str(MODEL_DIR / 'eval_metrics_table.png'), dpi=150, bbox_inches='tight')
plt.show()